In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col
)

In [0]:
storageAccount = "sourceskitkv"
containerName = "datalake"

historicalPath = f"abfss://{containerName}@{storageAccount}.dfs.core.windows.net/bronze/historical"

incrementalPath = f"abfss://{containerName}@{storageAccount}.dfs.core.windows.net/bronze/incremental"

bronzeOutput = f"abfss://{containerName}@{storageAccount}.dfs.core.windows.net/bronze/delta/jobs"


In [0]:
historicalDF = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load(historicalPath)
         .select("*", "_metadata")
)

In [0]:
incrementalDF = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load(incrementalPath)
         .select("*", "_metadata")
)

In [0]:
historicalDF = (
    historicalDF
        .withColumn("LoadType", lit("Historical"))
        .withColumn("IngestionTimestamp", current_timestamp())
        .withColumn("SourceFile", col("_metadata.file_path"))
        .drop("_metadata")
)

incrementalDF = (
    incrementalDF
        .withColumn("LoadType", lit("Incremental"))
        .withColumn("IngestionTimestamp", current_timestamp())
        .withColumn("SourceFile", col("_metadata.file_path"))
        .drop("_metadata")
)

In [0]:
bronzeDF = historicalDF.unionByName(incrementalDF)



In [0]:
bronzeDF.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("skitkartev.bronze.jobs")

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS skitkartev.bronze
""")

DataFrame[]

In [0]:
%sql
SELECT COUNT(*) AS BronzeRows
FROM skitkartev.bronze.jobs;

BronzeRows
506000
